# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pangakeerthi11/keerthi-flyrank-ml-internship-2026/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:

import os
from pathlib import Path
import subprocess

repo_path = Path("/content/keerthi-flyrank-ml-internship-2026")

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Pangakeerthi11/keerthi-flyrank-ml-internship-2026.git",
            str(repo_path)
        ],
        check=True
    )
else:
    print("Repository already exists.")

os.chdir(repo_path)

print("Current directory:", Path.cwd())
print("Repository cloned successfully!")


Repository already exists.
Current directory: /content/keerthi-flyrank-ml-internship-2026
Repository cloned successfully!


In [9]:

from pathlib import Path

print("Searching for FlyRank dataset...")

dataset_files = list(
    Path("/content/keerthi-flyrank-ml-internship-2026").rglob(
        "content_refresh_anonymized.csv"
    )
)

if dataset_files:
    for file in dataset_files:
        print("Dataset found:", file)
else:
    print("Dataset is not present in the cloned repository.")
    print("\nAvailable CSV files:")

    for file in Path.cwd().rglob("*.csv"):
        print(file)


Searching for FlyRank dataset...
Dataset found: /content/keerthi-flyrank-ml-internship-2026/data/raw/content_refresh_anonymized.csv


In [10]:

import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

if "trend_direction" in df.columns:
    print("\nLabel source: trend_direction")

    print("\nLabel distribution:")
    print(df["trend_direction"].value_counts(dropna=False))

    print("\nUnique label values:")
    print(df["trend_direction"].dropna().unique())

else:
    print("\ntrend_direction column not found.")


Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Label source: trend_direction

Label distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Unique 

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1
The first finding I selected from the FlyRank research paper is:
[Write the first finding exactly in your own words, based on the paper.]

**Methodology questions:**
- Where does the label used for this finding come from?
- Is the label based on an observed outcome, a manually assigned category, or a proxy?
- Does the validation design prevent the same client or related records from appearing in both training and testing?

### Finding 2
The second finding I selected from the FlyRank research paper is:
[Write the second finding exactly in your own words, based on the paper.]

**Methodology questions:**
- How was the target outcome measured?
- Was the validation performed on unseen clients or future observations?
- Does the validation design support the strength of the claim, or should the result be treated as directional evidence?

These questions are intended to understand how reliable and generalizable the findings are, not to dismiss the research.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

I will compare a random row-based split with a client-grouped split.

The random split is used as a reference. The grouped split keeps each client entirely in either the training set or the test set, which helps check whether the model performs well on clients it has not seen before.

I will keep the target definition, features, and model settings consistent across both evaluations. I will compare Precision@50, accuracy, and ROC-AUC where these metrics can be calculated.

A difference between the two results may indicate that the random split gives an overly optimistic estimate of performance.

In [11]:

import os
import sys
import subprocess
from pathlib import Path

# Set the FlyRank repository folder
repo_path = Path("/content/keerthi-flyrank-ml-internship-2026")
os.chdir(repo_path)

# Check the Week-5 preparation script
script_path = repo_path / "scripts" / "01_prepare_features.py"

if not script_path.exists():
    raise FileNotFoundError(f"Preparation script not found: {script_path}")

# Run feature preparation
result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=str(repo_path),
    capture_output=True,
    text=True
)

print("SCRIPT OUTPUT:")
print(result.stdout)

if result.stderr:
    print("\nERROR / ADDITIONAL OUTPUT:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Feature preparation failed. Check the output above.")

# Check generated processed files
processed_path = repo_path / "data" / "processed"
processed_files = list(processed_path.glob("*.csv"))

print("\nPROCESSED CSV FILES:")
for file in processed_files:
    print(file)

if not processed_files:
    print("No processed CSV was generated.")
else:
    print("\nFeature preparation completed successfully!")



SCRIPT OUTPUT:
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/keerthi-flyrank-ml-internship-2026/data/processed/refresh_feature_vector.csv


PROCESSED CSV FILES:
/content/keerthi-flyrank-ml-internship-2026/data/processed/refresh_feature_vector.csv

Feature preparation completed successfully!


In [12]:

import os
import sys
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    roc_auc_score,
    average_precision_score
)

# Set repository directory
repo_path = Path("/content/keerthi-flyrank-ml-internship-2026")
os.chdir(repo_path)

# Import the official feature configuration
sys.path.insert(0, str(repo_path / "scripts"))

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

# Load the prepared dataset
data_path = repo_path / "data/processed/refresh_feature_vector.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# Define target and client group
target_col = "is_declining_label"
group_col = "client_id"

required_cols = [target_col, group_col]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

# Use the official configured model features
feature_cols = [
    col for col in (
        list(MODEL_NUMERIC_FEATURES) +
        list(MODEL_CATEGORICAL_FEATURES)
    )
    if col in df.columns
]

# Exclude identifiers and target-related columns
feature_cols = [
    col for col in feature_cols
    if col not in [target_col, group_col, "trend_direction"]
]

if not feature_cols:
    raise ValueError("No configured model features found.")

X = df[feature_cols].copy()
y = df[target_col].astype(int)
groups = df[group_col]

numeric_features = [
    col for col in MODEL_NUMERIC_FEATURES if col in feature_cols
]

categorical_features = [
    col for col in MODEL_CATEGORICAL_FEATURES if col in feature_cols
]

print("Number of model features:", len(feature_cols))
print("Target distribution:")
print(y.value_counts())

# Build the same model for both validation approaches
def build_model():

    transformers = []

    if numeric_features:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

        transformers.append(
            ("numeric", numeric_pipeline, numeric_features)
        )

    if categorical_features:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        transformers.append(
            ("categorical", categorical_pipeline, categorical_features)
        )

    preprocessor = ColumnTransformer(transformers)

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ))
    ])

# Calculate Precision@50
def precision_at_k(y_true, scores, k=50):
    k = min(k, len(scores))
    top_indices = np.argsort(scores)[::-1][:k]
    return np.asarray(y_true)[top_indices].mean()

# Train and evaluate one split
def evaluate_split(train_idx, test_idx, split_name):

    model = build_model()

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    if y_train.nunique() < 2:
        raise ValueError(
            f"{split_name}: Training data contains only one target class."
        )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Split": split_name,
        "Training rows": len(train_idx),
        "Testing rows": len(test_idx),
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Precision@50": precision_at_k(
            y_test, probabilities, 50
        ),
        "ROC-AUC": (
            roc_auc_score(y_test, probabilities)
            if y_test.nunique() == 2 else np.nan
        ),
        "Average Precision": (
            average_precision_score(y_test, probabilities)
            if y_test.nunique() == 2 else np.nan
        )
    }

# BEFORE: Random row-based split
indices = np.arange(len(df))

random_train, random_test = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# AFTER: Client-grouped split
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train, group_test = next(
    group_split.split(X, y, groups=groups)
)

# Evaluate both approaches
before_results = evaluate_split(
    random_train, random_test, "Random row split"
)

after_results = evaluate_split(
    group_train, group_test, "Client-grouped split"
)

# Display comparison
comparison = pd.DataFrame([
    before_results,
    after_results
])

print("\nBEFORE AND AFTER VALIDATION RESULTS")
display(comparison.round(4))

# Verify client separation
shared_clients = (
    set(groups.iloc[group_train].dropna()) &
    set(groups.iloc[group_test].dropna())
)

print("\nClients shared between grouped train and test:", len(shared_clients))

assert len(shared_clients) == 0, "Client leakage detected!"

print("\nSection 2 model comparison completed successfully.")


Dataset shape: (30000, 52)
Number of model features: 26
Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

BEFORE AND AFTER VALIDATION RESULTS


,Split,Training rows,Testing rows,Accuracy,Precision,Precision@50,ROC-AUC,Average Precision
0,Random row split,24000,6000,0.6920,0.7089,0.90,0.7579,0.7684
1,Client-grouped split,23837,6163,0.5827,0.5900,0.54,0.6096,0.5896



Clients shared between grouped train and test: 0

Section 2 model comparison completed successfully.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Feature leakage audit

I will inspect the final feature set used by my model to identify possible data leakage.

I will check whether each feature is available at prediction time and whether it contains future information or information derived from the target.

The target column `is_declining_label` will not be used as an input feature. I will also verify that `trend_direction`, which is used to create the target, is excluded from the model inputs.

I will document suspicious features and verify their definitions before deciding whether they are safe to use.

In [13]:

import os
import sys
import subprocess
import pandas as pd
from pathlib import Path

# Step 1: Restore the FlyRank repository
repo_path = Path("/content/keerthi-flyrank-ml-internship-2026")

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Pangakeerthi11/keerthi-flyrank-ml-internship-2026.git",
            str(repo_path)
        ],
        check=True
    )

os.chdir(repo_path)

print("Repository ready:", repo_path)

# Step 2: Generate processed data if it is missing
data_path = repo_path / "data/processed/refresh_feature_vector.csv"

if not data_path.exists():
    script_path = repo_path / "scripts/01_prepare_features.py"

    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=str(repo_path),
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Feature preparation failed.")

# Step 3: Import official feature configuration
sys.path.insert(0, str(repo_path / "scripts"))

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

# Step 4: Load prepared dataset
df = pd.read_csv(data_path)

target_col = "is_declining_label"
group_col = "client_id"

# Step 5: Recreate model feature list
feature_cols = [
    col for col in (
        list(MODEL_NUMERIC_FEATURES) +
        list(MODEL_CATEGORICAL_FEATURES)
    )
    if col in df.columns
]

feature_cols = [
    col for col in feature_cols
    if col not in [target_col, group_col, "trend_direction"]
]

# Step 6: Check suspicious feature names
suspicious_terms = [
    "label",
    "target",
    "future",
    "outcome",
    "trend_direction",
    "declining"
]

leakage_candidates = [
    col for col in feature_cols
    if any(term in col.lower() for term in suspicious_terms)
]

print("\nDataset shape:", df.shape)
print("Number of model features:", len(feature_cols))

print("\nFinal model features:")
print(feature_cols)

print("\nPotential leakage candidates:")

if leakage_candidates:
    for col in leakage_candidates:
        print("REVIEW:", col)
else:
    print("No suspicious feature names found.")

# Step 7: Explicit checks
print("\nTarget included:", target_col in feature_cols)
print("trend_direction included:", "trend_direction" in feature_cols)
print("client_id included:", group_col in feature_cols)

assert target_col not in feature_cols
assert "trend_direction" not in feature_cols
assert group_col not in feature_cols

print("\nExplicit leakage checks passed.")
print("Section 3 name-based audit completed.")


Repository ready: /content/keerthi-flyrank-ml-internship-2026

Dataset shape: (30000, 52)
Number of model features: 26

Final model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Potential leakage candidates:
No suspicious feature names found.

Target included: False
trend_direction included: False
client_id included: False

Explicit leakage checks passed.
Section 3 name-based audit completed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

**Original claim:**

The Random Forest model accurately predicts which content will decline and can identify the best content to refresh.

**Rewritten claim:**

In the evaluated dataset, the Random Forest model showed measurable performance in identifying content labelled as declining. Its Precision@50 provides a directional indication of how useful the model may be for prioritizing content for review. This result should be treated as decision-support evidence, not proof that the model will predict future performance accurately for every client.

The claim is limited to the dataset and validation design used in this experiment.

In [14]:

import pandas as pd

# Use the actual results obtained in Section 2
comparison = pd.DataFrame([
    {
        "split": "Random row split",
        "precision_at_50": 0.90,
        "accuracy": 0.6920,
        "roc_auc": 0.7579
    },
    {
        "split": "Client-grouped split",
        "precision_at_50": 0.54,
        "accuracy": 0.5827,
        "roc_auc": 0.6096
    }
])

print("Model evaluation results:")

display(
    comparison[
        ["split", "precision_at_50", "accuracy", "roc_auc"]
    ].round(4)
)

print("\nInterpretation:")
print(
    "These metrics describe performance on the evaluated test data. "
    "They do not guarantee future results for every client."
)

print("\nSection 4 claim evaluation completed.")


Model evaluation results:


,split,precision_at_50,accuracy,roc_auc
0,Random row split,0.90,0.6920,0.7579
1,Client-grouped split,0.54,0.5827,0.6096



Interpretation:
These metrics describe performance on the evaluated test data. They do not guarantee future results for every client.

Section 4 claim evaluation completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.